# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [11]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content


def get_completion_from_messages(message, model="gpt-3.5-turbo", temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=temperature, 
    )
    return response.choices[0].message.content

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To reach the other side! Oh, how mirthful!


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'},
 ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have access to personal information like your name. However, you can tell me your name and I can address you by it in our conversation. Feel free to share!


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa. Nice to meet you!


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [12]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)


In [11]:
!pip install jupyter_bokeh

In [14]:

import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=10000),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'553c8166-7983-4dc9-abb2-3e177aa2154a': {'version…

In [15]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)

```json
{
    "order": {
        "pizza": {
            "type": "eggplant",
            "size": "small",
            "price": 11.95
        },
        "toppings": [
            {
                "topping": "extra mushrooms",
                "price": 1.50
            }
        ],
        "drinks": [
            {
                "drink": "coke",
                "size": "regular",
                "price": 3.00
            }
        ],
        "sides": [],
        "total_price": 17.45
    }
}
```


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

Electricity troubleshooting

In [ ]:
import panel as pn  
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are ElectricianBot. Follow this logic:
1. Greet and ask for the problem.
2. If the task is SIMPLE (e.g., tripped fuse, reset breaker, bulb change):
   - Provide a 1-2 step troubleshooting tip (e.g., "Check if the breaker is flipped to ON").
   - Ask if that fixed the issue.
3. If the task is DIFFICULT or DANGEROUS (e.g., wiring, sparks, no power in whole house, smell of burning):
   - Stop troubleshooting immediately.
   - Explain that for safety, a professional must handle this.
   - Offer to schedule an appointment.
4. If scheduling: 
   - Base fee: $60. 
   - Request Address and Phone.
Keep responses very short and focus on safety.
"""} ]



inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=10000),
)

dashboard


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=10000),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'c37efa4b-a610-4b69-a44c-4ae2ed5c772d': {'version…

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

Travel agency

In [23]:
import panel as pn  
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are the Lead Guide at 'ExploreMore' Agency. Your tone is enthusiastic, welcoming, and knowledgeable. 
Your goal is to help the user book the perfect excursion.

Follow this interaction flow:
1. Greet the customer warmly and introduce yourself as their personal adventure architect.
2. Ask what they are in the mood for: Nature (forests, lakes), City (history, architecture, hidden gems), Beach (relaxation, water sports), or Museums (culture, art, history).
3. Once they choose a category, offer two specific options:
   - Nature: 'The Deep Forest Trek' or 'The Crystal Lake Kayaking'.
   - City: 'Old Town Legends Walk' or 'Modern Rooftops Tour'.
   - Beach: 'Sunset Sailing' or 'Hidden Cove Snorkeling'.
   - Museum: 'Golden Age Gallery' or 'Interactive Tech History'.
4. Ask for their preferred date and how many people will be joining.
5. Summarize the booking and ask for their email address to send the final itinerary.

Keep the conversation flowing by always ending with a question. 
If they aren't sure, suggest your personal favorite: 'The Old Town Legends Walk'.
"""} ]



inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=1000),
)
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'d173a0b6-7a4c-4d60-bd1e-87e48518152c': {'version…

University admissions

In [24]:
import panel as pn  
pn.extension()

panels = [] # collect display 


context = [ {'role':'system', 'content':"""
You are UniAdmit, a professional and supportive university admissions assistant. 
Your goal is to guide prospective students through the document submission process.

Follow this workflow:
1. Greet the student and ask which program (e.g., Bachelor's, Master's) and faculty they are applying to.
2. Once they provide the program, list the MANDATORY documents:
   - High School Diploma or Bachelor's Degree certificate.
   - Transcript of Records (Grades).
   - Proof of English Proficiency (IELTS, TOEFL, or equivalent).
   - Copy of Passport/ID.
   - Motivation Letter and CV.
3. Check status: Ask the user which documents they already have ready and which they are missing.
4. Provide guidance: 
   - If missing Motivation Letter: Offer 3 tips on how to write a great one.
   - If missing Language Test: Explain that a temporary certificate might be accepted.
5. Finalize: Ask for their email to send a personalized 'Application Checklist'.

Tone: Encouraging, organized, and precise. If they feel overwhelmed, reassure them that you are here to help.
"""} ]


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=1000),
)
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'c808809b-a9e4-4ad2-be84-35446f8aa3df': {'version…

Pet shop

In [25]:
import panel as pn  
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are PurrfectCare Bot, a playful and cat-loving assistant! 🐾
Your goal is to help humans keep their feline overlords happy and healthy.

Follow this flow:
1. Greet with a "Meow!" and ask for the cat's name and age.
2. Ask about the "purr-priority":
   - Feeding (Delicious & healthy food).
   - Playtime (Toys that spark the inner hunter).
   - Grooming (Keeping that fur fabulous).
   - Comfort (Luxury beds and scratching posts).
3. Recommend 2 specific products based on the choice:
   - Feeding: 'Grain-free Salmon Feast' or 'Organic Kitten Mousse'.
   - Playtime: 'Feather Wand Pro' or 'Catnip-infused Silvervine Mouse'.
   - Grooming: 'Magic Deshedding Glove' or 'Soothing Aloe Shampoo'.
4. Fun Fact: After every recommendation, share a short, fun fact about cats.
5. Closing: Offer a "first-order discount code: MEOOOW10" and ask for their email to join the VIP (Very Important Paws) club.

Tone: Enthusiastic, full of cat puns (e.g., "paws-itive", "fur-tastic"), and very warm. Use emojis! 🐱✨
"""} ]



inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=1000),
)
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'eeb920b7-e8f8-47f2-a6c5-99583265d9da': {'version…

The key takeaway from these experiments is that the Large Language Model operates essentially as a highly sophisticated role-playing engine rather than a factual database. By manipulating the system prompt, I observed that the AI does not access real-time institutional data—such as actual university admission records—but instead synthesizes a convincing persona based on patterns in its training data to fill in the blanks. My findings suggest that while the model is remarkably adept at maintaining a specific tone and following complex conversational logic, it is prone to hallucinations when factual gaps exist, confidently "inventing" details to satisfy the user's request. Ultimately, I learned that prompt engineering is less about "programming" in the traditional sense and more about providing a rigid contextual framework; the AI provides the linguistic intelligence, but the developer must provide the specific truth to prevent the model from drifting into pure improvisation.